In [1]:
#Transformation JSON → JSON MongoDB propre
# Importer les bibliothèques
import json
from pathlib import Path
from datetime import datetime
import os


In [2]:
#   META STATIONS

STATIONS_META = {
    "ILAMAD25": {
        "name": "La Madeleine",
        "lat": 50.659,
        "lon": 3.07,
        "elevation": 23,
        "city": "La Madeleine",
        "hardware": "other",
        "software": "EasyWeatherPro_V5.1.6",
        "source": "Personal Station"
    },
    "IICHTE19": {
        "name": "WeerstationBS",
        "lat": 51.092,
        "lon": 2.999,
        "elevation": 15,
        "city": "Ichtegem",
        "hardware": "other",
        "software": "EasyWeatherV1.6.6",
        "source": "Personal Station"
    }
}


In [3]:
#   OUTILS
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save_json(path, content):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(content, f, indent=4, ensure_ascii=False)


def normalize_measure(record):
    """
    Normalise les champs météo possibles
    (gère noms différents selon fichiers)
    """
    def get(*keys):
        for k in keys:
            if k in record and record[k] not in ["", None]:
                return record[k]
        return None

    return {
        "timestamp": get("timestamp", "date", "datetime"),
        "temperature": get("temperature", "temp", "t"),
        "humidity": get("humidity", "hum"),
        "pressure": get("pressure", "pres"),
        "wind_speed": get("wind_speed", "wind"),
        "wind_gust": get("wind_gust"),
        "rain_rate": get("precip_rate", "rain_rate"),
        "rain_total": get("precip_accum", "rain_total"),
        "uv": get("uv"),
        "solar": get("solar")
    }


def build_station_doc(station_id, measures):
    meta = STATIONS_META.get(station_id, {})

    return {
        "station_id": station_id,
        "source": meta.get("source"),
        "name": meta.get("name"),
        "city": meta.get("city"),
        "hardware": meta.get("hardware"),
        "software": meta.get("software"),
        "location": {
            "latitude": meta.get("lat"),
            "longitude": meta.get("lon"),
            "elevation": meta.get("elevation"),
        },
        "data": measures
    }



In [4]:
#   TESTS QUALITÉ
def run_quality_checks(doc):
    problems = []

    # Station info
    if not doc["station_id"]:
        problems.append(" station_id manquant")

    if not doc["location"]["latitude"] or not doc["location"]["longitude"]:
        problems.append(" coordonnées absentes")

    # Données météo
    if "data" not in doc or len(doc["data"]) == 0:
        problems.append(" aucune donnée météo")

    # Types corrects
    temps = [d.get("timestamp") for d in doc["data"] if d.get("timestamp")]
    if len(temps) > 1 and len(temps) != len(set(temps)):
        problems.append(" timestamps dupliqués")

    if problems:
        print("\n PROBLÈMES TROUVÉS :")
        for p in problems:
            print(p)
    else:
        print(" Tests OK — structure propre")


In [5]:

#   TRAITEMENTS

def process_personal_station(file, station_id, out_name):
    print(f"\n---- Traitement station personnelle → {station_id} ----")

    raw = load_json(file)

    # Gestion des deux formats possibles
    if isinstance(raw, dict):
        measures_raw = raw.get("data", [])
    elif isinstance(raw, list):
        measures_raw = raw
    else:
        raise ValueError(f"Format JSON inattendu pour {file}")

    measures = [normalize_measure(rec) for rec in measures_raw]

    doc = build_station_doc(station_id, measures)
    save_json(out_name, doc)
    print(f" Sauvé → {out_name}")



def process_infoclimat(file):
    raw = load_json(file)

    # ---- STATIONS ----
    stations_out = []
    for s in raw["stations"]:
        stations_out.append({
            "station_id": s["id"],
            "name": s["name"],
            "source": "InfoClimat",
            "location": {
                "latitude": s["latitude"],
                "longitude": s["longitude"],
                "elevation": s["elevation"]
            },
            "license": s["license"]
        })

    save_json("output/infoclimat_mongo.json", stations_out)
    print(" Généré : output/infoclimat_mongo.json")

    # ---- MESURES ----
    #  InfoClimat.json ne contient PAS de mesures → on met vide proprement
    measures_out = []

    save_json("output/infoclimat_measures_mongo.json", measures_out)
    print(" Généré : output/infoclimat_measures_mongo.json")




In [6]:
#   MAIN

if __name__ == "__main__":
    print("===== TRANSFORMATION LANCÉE =====")

    process_personal_station(
        "data/la_madeleine.json",
        "ILAMAD25",
        "la_madeleine_mongo.json"
    )

    process_personal_station(
        "data/ichtegem.json",
        "IICHTE19",
        "ichtegem_mongo.json"
    )

    process_infoclimat("data/InfoClimat.json")

    print("\n FINI — fichiers prêts pour MongoDB")


===== TRANSFORMATION LANCÉE =====

---- Traitement station personnelle → ILAMAD25 ----
 Sauvé → la_madeleine_mongo.json

---- Traitement station personnelle → IICHTE19 ----
 Sauvé → ichtegem_mongo.json
 Généré : output/infoclimat_mongo.json
 Généré : output/infoclimat_measures_mongo.json

 FINI — fichiers prêts pour MongoDB
